# Phase 2: Multi-Class Classification Pipeline — KNN & Ensemble

This notebook implements the complete 10-class classification pipeline for the MNIST dataset.

**Required Improvements included:**
1. **Feature Engineering**: Using PCA (95% variance).
2. **Hyperparameter Tuning with Cross-Validation**: Implementing 3-Fold CV from scratch to find the best K.
3. **Learning Curves**: Plotting train vs validation accuracy to diagnose bias/variance tradeoff.
4. **Ensemble Methods**: Implementing a custom **Bagged KNN** ensemble classifier.
5. **Data Balancing**: Under-sampling all 10 classes to balance the dataset.


## 0. Imports & Setup

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.utils.mnist_features import load_mnist, split_data, normalize_data, build_features, class_distribution, balance_multi_classes
from src.utils.evaluation import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, per_class_accuracy
from src.models.knn_model import KNNClassifier, grid_search_knn

## 1. Load & Preprocess Data
We load the full 10-class MNIST dataset.

In [ ]:
# Load the full MNIST dataset
x_all, y_all = load_mnist('../../data/mnist.npz')
print(f'Full dataset shape: {x_all.shape}, Labels shape: {y_all.shape}')
print('Class distribution:', class_distribution(y_all))

### 1.1 Train / Test Split & Normalization


In [ ]:
# Split 80% train, 0% static val, 20% test
x_train, y_train, _, _, x_test, y_test = split_data(
    x_all, y_all, test_size=0.20, val_size=0.0, random_state=42
)
print(f'Train size: {len(y_train)} | Test size: {len(y_test)}')

# Balance the training data (Under-sampling)
print('\nBalancing multi-class training data...')
x_train, y_train = balance_multi_classes(x_train, y_train)
print(f'Balanced train size: {len(y_train)}')
print('Balanced distribution:', class_distribution(y_train))

# Normalize
x_train = normalize_data(x_train)
x_test  = normalize_data(x_test)


## 2. Feature Extraction
We use PCA (retaining 95% variance) to make KNN computation feasible on the full 60k dataset.

In [ ]:
# PCA Feature Extraction
# Note: val set is empty so we pass x_test as a dummy to avoid errors
train_f, _, test_f = build_features('pca', x_train, np.empty((0, 28, 28)), x_test, pca_components=0.95, random_state=42)
print(f'PCA features shape: {train_f.shape[1]} dimensions')

## 3. Hyperparameter Tuning (Cross-Validation) & Learning Curves
We use 3-Fold Cross-Validation across K values to find the best K for standard KNN.

In [ ]:
k_values = [1, 3, 5, 7, 9]
print('Running 3-Fold Cross-Validation with DISTANCE REUSE optimization...')

# Use optimized grid_search_knn that computes distances ONCE per fold
grid_results, best_params = grid_search_knn(k_values, train_f, y_train, cv=3)

# Convert grid_results to the same format
cv_results = {entry['params']['k']: {'val_accuracy': entry['val_accuracy'], 'train_accuracy': entry['train_accuracy']} for entry in grid_results}
best_k = best_params['k'] if best_params is not None else None

print(f'\nBest K = {best_k}')

### 3.1 Learning Curves (Bias/Variance Analysis)

In [ ]:
train_accs = [cv_results[k]['train_accuracy'] for k in k_values]
val_accs = [cv_results[k]['val_accuracy'] for k in k_values]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_values, train_accs, 'o-', label='Train Accuracy (Mean over CV)', linewidth=2, color='blue')
ax.plot(k_values, val_accs, 's-', label='Validation Accuracy (Mean over CV)', linewidth=2, color='orange')

ax.set_xlabel('K (Number of Neighbors)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Learning Curves: Train vs CV Validation Accuracy', fontsize=13, fontweight='bold')
ax.set_xticks(k_values)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('Analysis: Lower K (e.g., K=1) often shows high train accuracy (overfitting / low bias, high variance). As K increases, the model generalizes better, but too high K causes underfitting (high bias).')

## 4. Final Evaluation: Standard KNN vs Bagged Ensemble KNN
We train the best standard model and a new **Bagged Ensemble Model** to see if the ensemble improves performance.

In [ ]:
print(f'\n--- Training Standard KNN (K={best_k}) ---')
final_model = KNNClassifier(k=best_k)
final_model.fit(train_f, y_train)
std_results = test_best_model(final_model, test_f, y_test, average='macro')

print(f'\nTest Accuracy: {std_results["accuracy"]:.4f}')
print(f'Test F1 Score: {std_results["f1_score"]:.4f}')

In [ ]:
# Memory-safe Bagged KNN evaluation
from features.evaluation import per_class_accuracy

n_estimators = 5
sample_ratio = 0.6
rng = np.random.default_rng(42)

print(f"\n--- Training Bagged KNN ({n_estimators} estimators, sample_ratio={sample_ratio}) ---")

bagged_preds = []
for est_idx in range(n_estimators):
    sample_size = int(len(y_train) * sample_ratio)
    bootstrap_idx = rng.choice(len(y_train), size=sample_size, replace=True)

    model_b = KNNClassifier(k=best_k)
    model_b.fit(train_f[bootstrap_idx], y_train[bootstrap_idx])
    bagged_preds.append(model_b.predict(test_f))

# Majority vote across estimators
bagged_pred_matrix = np.vstack(bagged_preds)
final_bagged_pred = np.empty(test_f.shape[0], dtype=y_train.dtype)
for j in range(test_f.shape[0]):
    values, counts = np.unique(bagged_pred_matrix[:, j], return_counts=True)
    final_bagged_pred[j] = values[np.argmax(counts)]

bagged_results = {
    'accuracy': accuracy_score(y_test, final_bagged_pred),
    'precision': precision_score(y_test, final_bagged_pred, average='macro'),
    'recall': recall_score(y_test, final_bagged_pred, average='macro'),
    'f1_score': f1_score(y_test, final_bagged_pred, average='macro'),
    'confusion_matrix': confusion_matrix(y_test, final_bagged_pred),
    'per_class_accuracy': per_class_accuracy(y_test, final_bagged_pred)
}

print(f"Bagged Test Accuracy: {bagged_results['accuracy']:.4f}")
print(f"Bagged Test F1 Score: {bagged_results['f1_score']:.4f}")

### 4.1 Per-Class Accuracy (Recall) for the Best Model

In [ ]:
best_results = bagged_results if bagged_results['accuracy'] > std_results['accuracy'] else std_results
pca = best_results['per_class_accuracy']

print('Accuracy (Recall) for each individual class:')
for label in sorted(pca.keys()):
    print(f'  Digit {label}: {pca[label]*100:.2f}%')

### 4.2 Confusion Matrix (10x10)

In [ ]:
# Final evaluation on TEST set using the best K for each method
std_results = {}

for method in feature_methods:
    train_f, _, test_f = features[method]
    k = best_ks[method]
    print(f'\n===== {method.upper()} (K={k}) on TEST set =====')
    model = KNNClassifier(k=k)
    model.fit(train_f, y_train)

    def validate_data(X, y):
        if X.shape[0] != y.shape[0]:
            raise ValueError(f"Mismatch in number of samples: X has {X.shape[0]}, y has {y.shape[0]}")
        if X.ndim != 2:
            raise ValueError(f"Features X must be a 2D array, got {X.ndim}D")
        if y.ndim != 1:
            raise ValueError(f"Labels y must be a 1D array, got {y.ndim}D")
        if not np.all(np.isfinite(X)):
            raise ValueError("Features X contain non-finite values (NaN or Inf)")

    validate_data(test_f, y_test)
    print("Testing best model on the test set...")

    y_pred = model.predict(test_f)
    y_probas = model.predict_proba(test_f) if hasattr(model, 'predict_proba') else None

    result = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='macro'),
        'recall': recall_score(y_test, y_pred, average='macro'),
        'f1_score': f1_score(y_test, y_pred, average='macro'),
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'per_class_accuracy': per_class_accuracy(y_test, y_pred),
        'test_preds': y_pred,
    }

    if y_probas is not None:
        result['test_probas'] = y_probas

    std_results[method] = result
    print(f'  Accuracy:  {result["accuracy"]:.4f}')
    print(f'  Precision: {result["precision"]:.4f}')
    print(f'  Recall:    {result["recall"]:.4f}')
    print(f'  F1 Score:  {result["f1_score"]:.4f}')